# Domain-sequential streams: does T3's gain survive when domains arrive one after another?

**Question.** Every result so far comes from mixed streams (all domains shuffled together). In deployment, shifts arrive domain by domain, and Ramen's per-class queues carry one domain's entries into the next. Does the zero-shot-weighted active set (T3) still beat Ramen there, and what does the stream order do to Ramen itself?

**Hypothesis (pre-registered, `PREREGISTRATION.md`, written before any sequential run).** T3 − Ramen, domain-averaged, mean over seeds 0–4, is above the noise floor of the sequential stream on **all three** of ImageNet-C 5K, DomainNet and ImageNet-R/A/Sketch. Mixed-stream gains for reference: +0.62 / +0.24 / +0.59.

**Why this test.** It is the main untested deployment condition. Same images, configs and seeds as the mixed runs; only the order changes.

**Runs.**

| Name | Stream | What it is | Knows domain boundaries? |
|---|---|---|---|
| NoAdapt | — | CLIP zero-shot (stateless; the seed-0 mixed run, realigned) | — |
| Ramen | sequential | `cfg/<ds>/Ramen.yaml`, never reset | no |
| T3 | sequential | RamenActive `active_set: prob`, `temperature: 3`, never reset | no |
| Ramen-reset | single | Ramen reset at every domain start; same within-domain order | **yes (reference)** |
| Ramen (mixed), T3 (mixed) | mixed | the earlier runs, for comparison | no |

**Data and order.** ImageNet-C 5K: 15 corruptions × 5,000, standard order gaussian_noise → jpeg_compression. DomainNet: clipart → infograph → painting → quickdraw → real → sketch (596,006). ImageNet-R/A/Sketch: rendition → adversarial → sketch (88,389; 1,000-class head). One order per dataset (not varied).

**Seeds.** 0–4: the within-domain order and Ramen's selection randomness; the images are identical.

**How the runs were produced.** `bash run_queue.sh` (detached, one worker per GPU) runs `run_sequential.py <dataset> <seed> <run>`; the harness's new `tta_mode='sequential'` visits domains in order without resetting. Then `python sequential_analysis.py` → `results/sequential/*.csv`.

**What counts as a result.** Noise floor = 2 × (max − min) of sequential Ramen over seeds 0–4, per dataset (computed below). Secondary, with no pass/fail: the order effect on Ramen (sequential vs mixed, both minus NoAdapt), sequential Ramen vs Ramen-reset, and the adaptation lag after each domain switch (first 10 batches of a domain vs the rest).

**Amendment (before any sequential result).** Ramen's fp16 priorities freeze full queues after 65,520 samples, which in a sequential stream locks the memory on early domains. Ramen-fp32 and T3-fp32 (`priority_fp32: true`, eviction only) were added: DomainNet seeds 0–4, the other two seed 0 as a screen. On DomainNet, T3-fp32 − Ramen-fp32 is read with the same floor and rule as H1.

**Sections.** 1 Setup · 2 Data · 3 Sanity checks · 4 Primary result · 5 What the order does to Ramen · 6 Adaptation lag, memory composition, fp32 · 7 Per domain · 8 Interpretation · 9 Glossary

## 1. Setup

In [ ]:
import os, sys
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')
sys.path.insert(0, '..'); sys.path.insert(0, '../2026-10-25_stacking')
import numpy as np, pandas as pd
import stack, sequential_analysis as sa
pd.set_option('display.width', 220)
acc, g, lag, pdm = sa.load_csv('acc'), sa.load_csv(), sa.load_csv('lag'), sa.load_csv('per_domain')
FL = sa.floor(acc)
acc.groupby(['dataset', 'run']).seed.nunique().unstack()

## 2. The data the code ran on

In [ ]:
stack.data_table(sa.DATASETS)

In [ ]:
pd.DataFrame({stack.LABEL[ds]: {'domain order': ' → '.join(pd.read_pickle(sa.path(ds, 0, 'sequential', 'Ramen')).domain.cat.categories),
                                 'Ramen (seq.) per seed': ', '.join(f'{a:.2f}' for a in acc[(acc.dataset == ds) & (acc.run == 'Ramen')].acc),
                                 'noise floor (2 x range)': round(FL[ds], 3)} for ds in sa.DATASETS}).T

## 3. Sanity checks

- **Sequential loader.** A NoAdapt run in sequential mode (ImageNet-C, seed 0) must give the same images, labels and predictions as the mixed NoAdapt run, with domains visited in order.
- **Reset reference.** Ramen and Ramen-reset see the same order; they must agree exactly on the first domain and may differ only afterwards.
- **Code changes for this experiment** (both default to the old behaviour): `torch.cdist` forced onto its matmul path (the default crashed on DomainNet's 1-image last batch of infograph; bitwise identical wherever the default worked), and a `priority_fp32` switch. Re-running the mixed ImageNet-C Ramen seed 0 after each change must reproduce the cached run on 100% of images.
- **Memory replay.** The offline replay of Ramen's queue rule (used in section 6b) must reproduce RamenActive's logged queue contents.

In [ ]:
sa.sanity()

In [ ]:
sa.reset_sanity()

In [ ]:
sa.code_change_checks()

In [ ]:
sa.validate_replay()

## 4. Primary result (pre-registered)

In [ ]:
s = sa.summary(g, FL); sa.verdict(s)

In [ ]:
fig = sa.plot_primary(s)

In [ ]:
show = s.copy(); show['dataset'] = show.dataset.map(stack.LABEL)
show.set_index(['dataset', 'comparison']).astype({'gain mean': float, 'gain std': float, 'gain min': float, 'images': float}).round(2)

## 5. What the order does to Ramen

Domain-averaged accuracy (mean over seeds). Same images in every column.

In [ ]:
a = acc.groupby(['dataset', 'run']).acc.mean().unstack()[['NoAdapt', 'Ramen (mixed)', 'Ramen', 'Ramen-reset', 'T3 (mixed)', 'T3']]
a.columns = ['NoAdapt', 'Ramen mixed', 'Ramen sequential', 'Ramen-reset (seq.)', 'T3 mixed', 'T3 sequential']
a['Ramen gain: mixed'] = a['Ramen mixed'] - a.NoAdapt
a['Ramen gain: sequential'] = a['Ramen sequential'] - a.NoAdapt
a['Ramen gain: reset'] = a['Ramen-reset (seq.)'] - a.NoAdapt
a.index = a.index.map(stack.LABEL); a.round(2)

## 6. Adaptation lag after a domain switch

In [ ]:
fig = sa.plot_lag(lag)

In [ ]:
t = sa.lag_table(lag); t.index = t.index.set_levels(t.index.levels[0].map(stack.LABEL), level=0); t.round(2)  # pts vs NoAdapt

### 6b. What the memory holds: fp16 vs fp32 priorities (amendment; descriptive)

Ramen's arrival-counter priorities are fp16. Every insert after sample 65,520 gets priority inf, and a full queue stops updating once all its slots are inf. Below, the whole memory's domain composition at the end of each domain, replayed offline (exact, validated in section 3) from the filing classes logged by the sequential T3 run, seed 0. Also shown: the share of each domain's images that entered the memory.

In [ ]:
mc = pd.concat([sa.memory_composition(ds, 0) for ds in sa.DATASETS])
mc.assign(dataset=mc.dataset.map(stack.LABEL)).set_index(['dataset', 'priorities', 'after domain']).drop(columns='seed').round(1).dropna(axis=1, how='all')

### 6c. fp32 priorities (amendment)

T3-fp32 − Ramen-fp32 uses the same floor and rule as H1 (DomainNet seeds 0–4; ImageNet-C and R/A/Sketch seed 0 only, as a screen). Ramen-fp32 − Ramen shows what the frozen memory costs or gains.

In [ ]:
f = s[s.comparison.isin(['T3-fp32 - Ramen-fp32', 'Ramen-fp32 - Ramen', 'T3-fp32 - T3', 'Ramen-fp32 - NoAdapt', 'Ramen - NoAdapt', 'T3 - Ramen'])]
f.assign(dataset=f.dataset.map(stack.LABEL)).set_index(['dataset', 'comparison'])[['seeds', 'gain mean', 'gain std', 'gain min', 'floor (seq.)', 'above floor', 'CI > 0', 'images']].astype(
    {'gain mean': float, 'gain std': float, 'gain min': float, 'images': float}).round(2)

## 7. Per domain

Accuracy minus NoAdapt per domain (pts, mean over seeds), domains in stream order. The first domain starts from an empty memory in every stream run.

In [ ]:
for ds in sa.DATASETS:
    p = pdm[pdm.dataset == ds]
    order = list(pd.read_pickle(sa.path(ds, 0, 'sequential', 'Ramen')).domain.cat.categories)
    cols = [c for c in ['Ramen (mixed)', 'Ramen', 'Ramen-reset', 'T3', 'Ramen-fp32', 'T3-fp32'] if c in p and p[c].notna().all()]
    m = p.groupby('domain')[['NoAdapt'] + cols].mean().loc[order]
    out = m[cols].sub(m.NoAdapt, axis=0)
    out.columns = [{'Ramen (mixed)': 'Ramen mixed', 'Ramen': 'Ramen seq.', 'T3': 'T3 seq.'}.get(c, c) for c in cols]
    out['T3 - Ramen (seq.)'] = out['T3 seq.'] - out['Ramen seq.']
    print(stack.LABEL[ds]); display(out.round(2))

## 8. Interpretation

**H1 fails on DomainNet.** T3's gain over Ramen survives the sequential stream on two of three datasets:
- ImageNet-C **+0.52 ± 0.06** (floor 0.37, 5/5 above; mixed +0.62).
- ImageNet-R/A/Sketch **+0.66 ± 0.11** (floor 0.40, 5/5; mixed +0.59).
- DomainNet **−0.07 ± 0.02** (floor 0.13, 0/5 above, every per-seed CI below or across zero; mixed +0.24).
- With fp32 priorities (the amendment) DomainNet is also negative, **−0.12 ± 0.01** (0/5). So the DomainNet failure is not caused by the fp16 freeze. Under the pre-registered rule, T3 is **not** confirmed as a general improvement for domain-sequential streams.

**Where T3 loses on DomainNet.** With fp32, T3 − Ramen is negative in all six domains (−0.06 to −0.17), including clipart, the first domain, where the memory holds only clipart. In the mixed stream it was positive in five of six (+0.16 to +0.52). One reading, which is untested: T3 helps when the memory mixes domains (mixed streams; ImageNet-C and R/A/Sketch, where the sequential memory stays mixed, see below) and does nothing or hurts when the memory is one domain. The direct test is T3 with a reset at each domain start, vs Ramen-reset. That is **post-hoc** and was not run.

**The fp16 priority bug matters in sequential streams** (amendment; replayed memory composition, seed 0):
- DomainNet: the fp16 memory freezes during painting. Only 6.7% of quickdraw and 6.8% of real images ever enter it, and 0.3% of sketch; at the end it is 77% infograph + painting.
- Fixing it is worth **+0.22 ± 0.01** for Ramen (5/5 above the floor) and +0.17 for T3; quickdraw alone goes from +3.30 to +4.27 over NoAdapt.
- ImageNet-C and R/A/Sketch: no effect (seed 0: −0.00 and +0.00), as predicted, because their streams barely pass the 65,520 point.
- Practical consequence: any non-stationary-stream claim about Ramen should use fp32 priorities, and published Ramen numbers on long sequential streams are partly measuring this bug.

**Carrying memory across domains hurts Ramen everywhere.** Ramen-reset, which knows the boundaries, beats sequential Ramen:
- ImageNet-C **+0.82 ± 0.05**: 0/5 seeds favour sequential.
- DomainNet +0.27, or about +0.04 against fp32 Ramen (56.96 vs 56.92).
- R/A/Sketch +0.16, below the floor.

The ImageNet-C case is a capacity effect, not a bug. Its 75 slots × 1,000 classes exceed the 5,000 images per corruption, so nothing is evicted and old corruptions pile up. At the end of corruption n only about 1/n of the memory is the current one (9.5% at jpeg_compression). The largest per-domain losses from not resetting are on late corruptions: elastic_transform −3.5, frost −2.1, glass_blur −1.8, jpeg_compression −1.6.

**Order effect on Ramen (secondary 1).** Sequential vs mixed (each minus NoAdapt): ImageNet-C +5.89 vs +5.74, R/A/Sketch +1.63 vs +1.48, DomainNet +1.32 vs +1.44 (fp32 +1.54). Apart from the fp16 freeze, Ramen does about as well, or slightly better, when domains arrive in order.

**Adaptation lag (secondary 3).** In the first 10 batches of a domain the gain over NoAdapt is lower than later: ImageNet-C Ramen +4.50 vs +6.23, DomainNet +0.87 vs +1.47. Ramen-reset starts from an empty memory, yet on ImageNet-C it is *better* early (+5.79). The stale memory costs more than an empty one. On DomainNet it is slightly worse early (+0.69) and better later (+1.85). R/A/Sketch shows no lag.

**Caveats.** One domain order per dataset. The fp32 screens on ImageNet-C and R/A/Sketch are seed 0 only. The "T3 needs a mixed memory" reading is a hypothesis from per-domain patterns.

**What this changes.**
1. T3 is confirmed on three mixed streams and two of three sequential ones; it is not a universal improvement. DomainNet, the largest stream and the one with the largest memory (300 per class, top-10, β = 5), is where it fails.
2. Two findings about Ramen that matter for any continual-TTA claim: the fp16 priority freeze, and memory staleness across domains when capacity exceeds the per-domain volume (ImageNet-C −0.82). A boundary-free fix for staleness (e.g. evicting by age, or a capacity matched to the stream rate) is a candidate method change. Its oracle ceiling, from Ramen-reset, is +0.82 on ImageNet-C and +0.27 on DomainNet.


## 9. Glossary

| Term | Meaning |
|---|---|
| Ramen | CLIP zero-shot + per-class memory of (feature, entropy gradient); one signSGD step per batch from retrieved entries, predict, reset the weights (the memory persists) |
| T3 | Ramen with a zero-shot-weighted active set: each class queue's vote weighted by the query's zero-shot probability at temperature 3 |
| mixed stream | all domains shuffled into one stream |
| sequential stream | domains one after another, each shuffled within itself; the method is never told where a domain starts |
| Ramen-reset | Ramen whose memory is emptied at every domain start (needs the boundaries; a reference, not a fair competitor) |
| adaptation lag | how long after a domain switch the gain over zero-shot takes to recover |
| fp16 / fp32 priorities | Ramen evicts the lowest arrival counter from a full queue; in fp16 the counter becomes inf after 65,520 samples and full queues stop changing (Ramen as published); fp32 keeps them updating |
| noise floor | 2 × (max − min) of sequential Ramen's domain-averaged accuracy over seeds 0–4 |
| domain-averaged accuracy | mean of per-domain accuracies |
| fixed / broken | images the reference gets wrong and the variant right / and the reverse |
| cluster bootstrap CI | 95% interval from resampling images (Poisson weights; base image on ImageNet-C), per seed |